In [ ]:
# Resolve the repository root so data/ and output/ paths work from any working directory
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "data").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

# Start

Making the data set hies_gdp_ntl_master

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from scipy.stats import spearmanr

In [2]:
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
sns.set_style("whitegrid")
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_style("whitegrid")

# Data reading

In [3]:
gdp = pd.read_csv(str(ROOT / "data/GDP_per_capita/Province_GDP_per_capita.csv"))

In [4]:
ntl = pd.read_csv(str(ROOT / "data/Iran_Province_NTL_Annual/Iran_Annual_NTLperCapita_Master_2012_2024.csv"))

In [5]:
exp = pd.read_excel(
    str(ROOT / "data/Validation/HIES_exp_per_capita_provincial_real.xlsx")
)

In [6]:
print("=== Expenditure ===")
print(exp.shape)
print(exp.columns.tolist())
print(exp.head(3))

print("\n=== NTL ===")
print(ntl.shape)
print(ntl.columns.tolist())
print(ntl.head(3))

print("\n=== GDP ===")
print(gdp.shape)
print(gdp.columns.tolist())
print(gdp.head(3))

=== Expenditure ===
(448, 5)
['Province', 'avg_exp_per_capita', 'Year', 'CPI', 'real_exp_per_capita']
  Province  avg_exp_per_capita  Year        CPI  real_exp_per_capita
0   Alborz        3.368149e+07  1390  12.344935         2.728365e+08
1  Ardabil        3.492718e+07  1390  12.017574         2.906342e+08
2  Bushehr        2.858612e+07  1390  11.882474         2.405738e+08

=== NTL ===
(403, 7)
['Area', 'Year', 'Year_Shamsi_NTL', 'Annual_Sum', 'Annual_Mean', 'Population', 'NTL_Per_Capita']
                     Area  Year  Year_Shamsi_NTL    Annual_Sum  Annual_Mean  \
0                    Yazd  2012             1391  37091.363543     0.242350   
1  Sistan and Baluchestan  2012             1391  39705.314040     0.196490   
2                  Kerman  2012             1391  54321.302014     0.262493   

   Population  NTL_Per_Capita  
0     1086569        0.034136  
1     2580705        0.015385  
2     2978370        0.018239  

=== GDP ===
(341, 5)
['نام استان', 'Year', 'GDP', 'Popula

# Data Cleaning

In [7]:
# ============================================================
# Step 1: Clean and standardize each dataset
# ============================================================

# --- Expenditure ---
exp_clean = exp.copy()
exp_clean = exp_clean.rename(columns={'Province': 'Province_EN'})
# Remove National row if exists
exp_clean = exp_clean[exp_clean['Province_EN'] != 'National'].reset_index(drop=True)
print("Exp provinces:", sorted(exp_clean['Province_EN'].unique()))
print("Exp years:", sorted(exp_clean['Year'].unique()))

Exp provinces: ['Alborz', 'Ardabil', 'Bushehr', 'Chaharmahal and Bakhtiari', 'East Azerbaijan', 'Fars', 'Gilan', 'Golestan', 'Hamadan', 'Hormozgan', 'Ilam', 'Isfahan', 'Kerman', 'Kermanshah', 'Khuzestan', 'Kohgiluyeh & Boyerahmad', 'Kurdistan', 'Lorestan', 'Markazi', 'Mazandaran', 'North Khorasan', 'Qazvin', 'Qom', 'Razavi Khorasan', 'Semnan', 'Sistan and Baluchestan', 'South Khorasan', 'Tehran', 'West Azerbaijan', 'Yazd', 'Zanjan']
Exp years: [np.int64(1390), np.int64(1391), np.int64(1392), np.int64(1393), np.int64(1394), np.int64(1395), np.int64(1396), np.int64(1397), np.int64(1398), np.int64(1399), np.int64(1400), np.int64(1401), np.int64(1402), np.int64(1403)]


In [8]:
ntl_clean = ntl.copy()
ntl_clean = ntl_clean.rename(columns={
    'Area': 'Province_EN',
    'Year_Shamsi_NTL': 'Shamsi_Year'
})
ntl_clean = ntl_clean[['Province_EN', 'Shamsi_Year', 'Annual_Sum', 'Annual_Mean', 'NTL_Per_Capita']]
ntl_clean = ntl_clean.rename(columns={'Shamsi_Year': 'Year'})

print(ntl_clean.columns.tolist())
print("NTL years:", sorted(ntl_clean['Year'].unique()))

['Province_EN', 'Year', 'Annual_Sum', 'Annual_Mean', 'NTL_Per_Capita']
NTL years: [np.int64(1391), np.int64(1392), np.int64(1393), np.int64(1394), np.int64(1395), np.int64(1396), np.int64(1397), np.int64(1398), np.int64(1399), np.int64(1400), np.int64(1401), np.int64(1402), np.int64(1403)]


In [9]:
# NTL cleaning — این خط را جایگزین کن:
ntl_clean = ntl[['Area', 'Year_Shamsi_NTL', 'Annual_Sum', 'Annual_Mean', 'NTL_Per_Capita', 'Population']].copy()
ntl_clean = ntl_clean.rename(columns={
    'Area': 'Province_EN',
    'Year_Shamsi_NTL': 'Year'
})

print(ntl_clean.columns.tolist())
print(ntl_clean.head(3))

['Province_EN', 'Year', 'Annual_Sum', 'Annual_Mean', 'NTL_Per_Capita', 'Population']
              Province_EN  Year    Annual_Sum  Annual_Mean  NTL_Per_Capita  \
0                    Yazd  1391  37091.363543     0.242350        0.034136   
1  Sistan and Baluchestan  1391  39705.314040     0.196490        0.015385   
2                  Kerman  1391  54321.302014     0.262493        0.018239   

   Population  
0     1086569  
1     2580705  
2     2978370  


In [10]:
print(gdp['نام استان'].unique())

['مرکزی' 'گیلان' 'مازندران' 'آذربایجان شرقی' 'آذربایجان غربی' 'کرمانشاه'
 'خوزستان' 'فارس' 'کرمان' 'خراسان رضوی' 'اصفهان' 'سیستان و بلوچستان'
 'کردستان' 'همدان' 'چهارمحال و بختیاری' 'لرستان' 'ایلام'
 'کهگیلویه و بویراحمد' 'بوشهر' 'زنجان' 'سمنان' 'یزد' 'هرمزگان' 'تهران'
 'اردبیل' 'قم' 'قزوین' 'گلستان' 'خراسان شمالی' 'خراسان جنوبی' 'البرز']


In [11]:
fa_to_en = {
    'مرکزی': 'Markazi', 'گیلان': 'Gilan', 'مازندران': 'Mazandaran',
    'آذربایجان شرقی': 'East Azerbaijan', 'آذربایجان غربی': 'West Azerbaijan',
    'کرمانشاه': 'Kermanshah', 'خوزستان': 'Khuzestan', 'فارس': 'Fars',
    'کرمان': 'Kerman', 'خراسان رضوی': 'Razavi Khorasan', 'اصفهان': 'Isfahan',
    'سیستان و بلوچستان': 'Sistan and Baluchestan', 'کردستان': 'Kurdistan',
    'همدان': 'Hamadan', 'چهارمحال و بختیاری': 'Chaharmahal and Bakhtiari',
    'لرستان': 'Lorestan', 'ایلام': 'Ilam',
    'کهگیلویه و بویراحمد': 'Kohgiluyeh & Boyerahmad', 'بوشهر': 'Bushehr',
    'زنجان': 'Zanjan', 'سمنان': 'Semnan', 'یزد': 'Yazd',
    'هرمزگان': 'Hormozgan', 'تهران': 'Tehran', 'اردبیل': 'Ardabil',
    'قم': 'Qom', 'قزوین': 'Qazvin', 'گلستان': 'Golestan',
    'خراسان شمالی': 'North Khorasan', 'خراسان جنوبی': 'South Khorasan',
    'البرز': 'Alborz',
}

In [12]:
# GDP cleaning — این خط را جایگزین کن:
gdp_clean = gdp.copy()
gdp_clean['Province_EN'] = gdp_clean['نام استان'].map(fa_to_en)
gdp_clean = gdp_clean[['Province_EN', 'Year', 'GDP', 'Population', 'GDP_per_Capita']]

missing = gdp[gdp['نام استان'].map(fa_to_en).isnull()]['نام استان'].unique()
print("Missing translations:", missing)
print("GDP provinces:", sorted(gdp_clean['Province_EN'].dropna().unique()))
print("GDP years:", sorted(gdp_clean['Year'].unique()))

Missing translations: []
GDP provinces: ['Alborz', 'Ardabil', 'Bushehr', 'Chaharmahal and Bakhtiari', 'East Azerbaijan', 'Fars', 'Gilan', 'Golestan', 'Hamadan', 'Hormozgan', 'Ilam', 'Isfahan', 'Kerman', 'Kermanshah', 'Khuzestan', 'Kohgiluyeh & Boyerahmad', 'Kurdistan', 'Lorestan', 'Markazi', 'Mazandaran', 'North Khorasan', 'Qazvin', 'Qom', 'Razavi Khorasan', 'Semnan', 'Sistan and Baluchestan', 'South Khorasan', 'Tehran', 'West Azerbaijan', 'Yazd', 'Zanjan']
GDP years: [np.int64(1390), np.int64(1391), np.int64(1392), np.int64(1393), np.int64(1394), np.int64(1395), np.int64(1396), np.int64(1397), np.int64(1398), np.int64(1399), np.int64(1400)]


# Bulding the master Data

in this part I make a master data and in there is every thing I need from now on 

In [13]:
# ============================================================
# Step 2: Build master dataset
# ============================================================

# Start with expenditure as the base (widest year coverage)
df_master = exp_clean[['Province_EN', 'Year', 'avg_exp_per_capita', 'real_exp_per_capita']].copy()

# Merge NTL
df_master = pd.merge(
    df_master,
    ntl_clean[['Province_EN', 'Year', 'Annual_Sum', 'Annual_Mean', 'NTL_Per_Capita', 'Population']],
    on=['Province_EN', 'Year'],
    how='left'
)


# Merge GDP
df_master = pd.merge(
    df_master,
    gdp_clean[['Province_EN', 'Year', 'GDP', 'GDP_per_Capita']],
    on=['Province_EN', 'Year'],
    how='left'
)


print(df_master.shape)
print(df_master.isnull().sum())
print(df_master.head())

(434, 10)
Province_EN             0
Year                    0
avg_exp_per_capita      0
real_exp_per_capita     0
Annual_Sum             31
Annual_Mean            31
NTL_Per_Capita         31
Population             31
GDP                    93
GDP_per_Capita         93
dtype: int64
                 Province_EN  Year  avg_exp_per_capita  real_exp_per_capita  \
0                     Alborz  1390        3.368149e+07         2.728365e+08   
1                    Ardabil  1390        3.492718e+07         2.906342e+08   
2                    Bushehr  1390        2.858612e+07         2.405738e+08   
3  Chaharmahal and Bakhtiari  1390        3.747039e+07         3.921692e+08   
4            East Azerbaijan  1390        3.230443e+07         3.163196e+08   

   Annual_Sum  Annual_Mean  NTL_Per_Capita  Population           GDP  \
0         NaN          NaN             NaN         NaN  1.489476e+09   
1         NaN          NaN             NaN         NaN  5.856598e+08   
2         NaN          NaN

In [15]:
# Log transforms
for col in ['Annual_Sum', 'Annual_Mean', 'NTL_Per_Capita', 
            'avg_exp_per_capita', 'real_exp_per_capita', 'GDP', 'GDP_per_Capita']:
    df_master[f'log_{col}'] = np.log(df_master[col])

# Save
import os
out_dir = str(ROOT / "data/Validation")

df_master.to_excel(os.path.join(out_dir, "hies_ntl_gdp_master.xlsx"), index=False)
df_master.to_stata(os.path.join(out_dir, "hies_ntl_gdp_master.dta"), write_index=False, version=118)

print("Saved.")
print(df_master.shape)
print(df_master.columns.tolist())

Saved.
(434, 17)
['Province_EN', 'Year', 'avg_exp_per_capita', 'real_exp_per_capita', 'Annual_Sum', 'Annual_Mean', 'NTL_Per_Capita', 'Population', 'GDP', 'GDP_per_Capita', 'log_Annual_Sum', 'log_Annual_Mean', 'log_NTL_Per_Capita', 'log_avg_exp_per_capita', 'log_real_exp_per_capita', 'log_GDP', 'log_GDP_per_Capita']
